# 🏠 التنبؤ بأسعار العقارات (House Price Prediction — Regression)
### مشروع B2 — مسار تعلم الآلة (Machine Learning Track)

---
## 🎯 المشكلة التجارية (Business Problem)
منصة عقارات عايزة **أداة تقدير تلقائي لسعر البيت** (زي Zillow's Zestimate) عشان تساعد البائعين
والمشترين يحطّوا سعر عادل، وتسرّع التقييم بدل المثمّن البشري.

**نوع المشكلة:** انحدار (Regression) — التنبؤ بقيمة رقمية مستمرة (السعر).

## 📦 ما الذي يثبته المشروع
معالجة القيم المفقودة · **تصحيح الالتواء (Skewness) باللوغاريتم** · Pipeline احترافي ·
**الانحدار المنتظم (Ridge/Lasso)** · Gradient Boosting · **تحليل البواقي (Residual Analysis)** · تفسير المعاملات.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b2_house_prices"          # مسار المشروع داخل portfolio/
PKGS    = ["xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| الانحدار الخطي وافتراضاته | ISLR (ch.3) | الأساس — لازم تفهم قبل أي حاجة |
| **التواء الهدف + Log transform** | Feature Eng. for ML (Zheng) | الأسعار ملتوية يمين → اللوغاريتم بيحسّن النموذج |
| القيم المفقودة (Imputation) | Géron (ch.2) | بيانات الواقع دايماً ناقصة |
| **الانتظام (Ridge / Lasso)** | ISLR (ch.6) | منع الـ overfitting + اختيار الميزات |
| Pipeline & ColumnTransformer | Géron (ch.2) | منع التسريب وتنظيم المعالجة |
| مقاييس الانحدار (RMSE, MAE, R²) | ISLR (ch.3) | تقييم الخطأ بوحدات السعر الحقيقية |
| **تحليل البواقي (Residuals)** | ISLR (ch.3) | كشف هل النموذج متحيّز أو ناقص ميزات |

> 🎯 **بيُستخدم في الواقع:** تقييم العقارات، تسعير السيارات المستعملة، تقدير قيمة الأصول، التأمين.


## 0️⃣ المكتبات

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style('whitegrid'); np.random.seed(42)
print('ready ✓')

ready ✓


## 1️⃣ تحميل واستكشاف البيانات (EDA)

In [2]:
df = pd.read_csv('data/house_prices_data.csv')
print('Shape:', df.shape)
print('Missing:\n', df.isna().sum()[df.isna().sum() > 0])
df.describe().T[['mean','std','min','max']]

Shape: (1500, 8)
Missing:
 square_footage    45
dtype: int64


,mean,std,min,max
square_footage,2031.273814,592.546808,600.0,4311.60
bedrooms,3.878000,1.258639,1.0,6.00
bathrooms,1.505667,0.477095,1.0,3.00
year_built,1987.106000,21.550539,1950.0,2023.00
garage_spaces,1.387333,0.887223,0.0,3.00
sale_price,491190.514727,175410.065908,82835.7,1039036.82


In [3]:
corr = df.corr(numeric_only=True)['sale_price'].sort_values(ascending=False)
print('Correlation with price:\n', corr.round(2))
fig, ax = plt.subplots(1,2, figsize=(13,4))
sns.scatterplot(data=df, x='square_footage', y='sale_price', ax=ax[0], alpha=.4)
ax[0].set_title('Price vs Square Footage')
sns.boxplot(data=df, x='neighborhood', y='sale_price', ax=ax[1]); ax[1].set_title('Price by Neighborhood')
plt.tight_layout(); plt.show()

Correlation with price:
 sale_price        1.00
square_footage    0.61
bedrooms          0.54
bathrooms         0.43
year_built        0.12
garage_spaces     0.08
Name: sale_price, dtype: float64


C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_23468\2152968693.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2️⃣ تصحيح التواء الهدف (Target Skewness → Log) 📐
أسعار العقارات **ملتوية لليمين** (قليل من البيوت غالية جداً). الانحدار الخطي بيفترض توزيع متماثل،
فبنطبّق `log1p` على السعر — ده بيحسّن الأداء بشكل ملحوظ. (نرجّع بـ `expm1` عند التقييم).

In [4]:
fig, ax = plt.subplots(1,2, figsize=(12,3.5))
sns.histplot(df['sale_price'], kde=True, ax=ax[0]); ax[0].set_title(f"Original (skew={df['sale_price'].skew():.2f})")
sns.histplot(np.log1p(df['sale_price']), kde=True, ax=ax[1], color='green')
ax[1].set_title(f"Log (skew={np.log1p(df['sale_price']).skew():.2f})")
plt.tight_layout(); plt.show()

C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_23468\1264207282.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 3️⃣ هندسة الميزات والمعالجة (Pipeline)

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df['house_age'] = 2024 - df['year_built']                 # ميزة جديدة أوضح من سنة البناء
X = df.drop(columns=['sale_price', 'house_id', 'year_built'])
y = np.log1p(df['sale_price'])                            # الهدف باللوغاريتم

num = ['square_footage','bedrooms','bathrooms','garage_spaces','house_age']
cat = ['neighborhood']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat),
])
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (1200, 6) | Test: (300, 6)


## 4️⃣ مقارنة الموديلات بالـ Cross-Validation (RMSE بوحدات السعر)

In [6]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_predict

def rmse_price(y_log_true, y_log_pred):
    return np.sqrt(np.mean((np.expm1(y_log_true) - np.expm1(y_log_pred))**2))

models = {
    'Linear': LinearRegression(),
    'Ridge':  Ridge(alpha=1.0),
    'Lasso':  Lasso(alpha=0.001),
    'RandomForest': RandomForestRegressor(n_estimators=300, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, random_state=42),
}
for name, m in models.items():
    pipe = Pipeline([('pre', pre), ('m', m)])
    pred = cross_val_predict(pipe, X_train, y_train, cv=5)
    print(f'{name:14} CV RMSE = ${rmse_price(y_train, pred):,.0f}')

Linear         CV RMSE = $28,750
Ridge          CV RMSE = $28,669


Lasso          CV RMSE = $28,372


RandomForest   CV RMSE = $29,489


XGBoost        CV RMSE = $23,631


## 5️⃣ تدريب أفضل موديل وتقييمه على الاختبار

In [7]:
from sklearn.metrics import mean_absolute_error, r2_score
best = Pipeline([('pre', pre), ('m', models['XGBoost'])])
best.fit(X_train, y_train)
pred_log = best.predict(X_test)
pred, true = np.expm1(pred_log), np.expm1(y_test)
print(f'Test RMSE = ${rmse_price(y_test, pred_log):,.0f}')
print(f'Test MAE  = ${mean_absolute_error(true, pred):,.0f}')
print(f'Test R²   = {r2_score(true, pred):.3f}')

Test RMSE = $20,823
Test MAE  = $16,321
Test R²   = 0.985


## 6️⃣ تحليل البواقي (Residual Analysis) 🔍
البواقي = (الحقيقي − المتوقع). المفروض تكون **عشوائية حول الصفر**. أي نمط فيها = الموديل بيفوّت حاجة.

In [8]:
resid = true - pred
fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].scatter(pred, resid, alpha=.4); ax[0].axhline(0, color='red', ls='--')
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Residual'); ax[0].set_title('Residuals vs Predicted')
sns.histplot(resid, kde=True, ax=ax[1]); ax[1].set_title('Residual distribution')
plt.tight_layout(); plt.show()

C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_23468\1364136410.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 7️⃣ أهمية الميزات (Feature Importance)

In [9]:
feat = best.named_steps['pre'].get_feature_names_out()
imp = pd.Series(best.named_steps['m'].feature_importances_, index=feat).sort_values()
imp.plot(kind='barh', figsize=(7,4), title='XGBoost Feature Importance'); plt.tight_layout(); plt.show()

C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_23468\3519039171.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  imp.plot(kind='barh', figsize=(7,4), title='XGBoost Feature Importance'); plt.tight_layout(); plt.show()


## 8️⃣ الخلاصة والتوصيات (Conclusion)
- **النتيجة:** XGBoost حقّق أقل RMSE وأعلى R² — يقدر يقدّر السعر بدقة كويسة.
- **أهم العوامل:** المساحة (square footage)، الحي (neighborhood)، عدد الحمامات — متوافق مع منطق السوق.
- **اللوغاريتم:** تحويل الهدف حسّن الأداء وخلّى البواقي أقرب للتوزيع الطبيعي.
- **التوصية:** استخدام الموديل كأداة تقدير أولي + مراجعة بشرية للحالات الشاذة (بواقي كبيرة).
- **الخطوة القادمة:** إضافة ميزات الموقع الدقيق، المسافة للخدمات، وبيانات السوق الزمنية.

> ✅ **اللي اتعلمته:** EDA، تصحيح الالتواء، Imputation، Pipeline، الانتظام، GBM، وتحليل البواقي.
